# Part 2 - Fine-tuning Stable Diffusion 1.5 with LoRA (rank 4 vs rank 32)

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

In [ ]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 2.1 Data preparation

In [ ]:
from p2_finetune import data_prep, prompts as P
stats = data_prep.clean_and_split(out_dir="work_p2/data"); print(stats)
test = data_prep.read_split("work_p2/data", "test"); train = data_prep.read_split("work_p2/data", "train")
PROMPTS = P.make_prompts([c for _, c in test], "prompts.json")   # commit prompts.json before training ends
pd.DataFrame(PROMPTS).head(20)

Dataset: Naruto BLIP captions (square-cropped to 512x512 - cropping can cut characters). Limitations/biases: (1) one franchise, mostly front-facing characters, so poor generalisation; (2) BLIP captions are short and sometimes wrong (e.g. 'a man with a sword'), which limits prompt alignment.

## 2.2 Two training runs (same data, steps, batch size, lr, seed)

In [ ]:
from p2_finetune.train_lora import train
info1 = train("lora_r4", 4, "work_p2/data")
info2 = train("lora_r32", 32, "work_p2/data")
pd.DataFrame([info1, info2]).set_index("run")[["rank", "trainable_params", "learning_rate", "steps", "batch_size", "peak_gpu_mem_gb", "train_seconds", "gpu_hours"]]

In [ ]:
# push to Hugging Face (needs HF_TOKEN in the environment; set repo ids first)
# from p2_finetune.hf_upload import push_adapter
# push_adapter("work_p2/runs/lora_r4",  "<user>/sd15-naruto-lora-r4")
# push_adapter("work_p2/runs/lora_r32", "<user>/sd15-naruto-lora-r32", results_text="see report")

## 2.3 Generation: 20 prompts x 2 seeds x 3 models = 120 images

In [ ]:
from p2_finetune.generate import generate_all, SETTINGS
print(SETTINGS)
META = generate_all({"base": None, "run1": "work_p2/runs/lora_r4", "run2": "work_p2/runs/lora_r32"}, PROMPTS)
len(META)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
sel = [p for p in PROMPTS if p["type"] == "in_domain"][:4] + [p for p in PROMPTS if p["type"] == "out_of_domain"][:2] + [p for p in PROMPTS if p["type"] == "compositional"][:2]
fig, ax = plt.subplots(len(sel), 3, figsize=(9, 3 * len(sel)))
for r, p in enumerate(sel):
    for c, m in enumerate(["base", "run1", "run2"]):
        ax[r, c].imshow(Image.open(f"work_p2/images/{m}/{p['id']}_s0.png")); ax[r, c].axis("off")
        ax[r, c].set_title(f"{m}: {p['prompt'][:34]}", fontsize=6)
plt.tight_layout(); plt.savefig("p2_grid.png", dpi=120); plt.show()

In [ ]:
# upload to a Hugging Face dataset (prompt, seed, model in metadata)
# from p2_finetune.hf_upload import push_images; push_images(META, "<user>/sd15-naruto-lora-generations")

## 2.4 Evaluation: KID + CLIP (primary), LPIPS diversity (secondary), efficiency

In [ ]:
from p2_finetune import evaluate as E
clip = E.Clip()
test_paths = [p for p, _ in test]; train_paths = [p for p, _ in train]
res = pd.DataFrame([E.summarize_model(clip, META, m, test_paths) for m in ["base", "run1", "run2"]]).set_index("model")
res["gpu_hours"] = [None, info1["gpu_hours"], info2["gpu_hours"]]
res["peak_gpu_mem_gb"] = [None, info1["peak_gpu_mem_gb"], info2["peak_gpu_mem_gb"]]
res

KID is computed from 20 in-domain generations vs the test images, so it is noisy (see std). Fill in which run is final after looking at the table and the human evaluation.

In [ ]:
FINAL = "run1"   # choose after reviewing results; justify in the report
from p2_finetune.human_eval import build_sheet
build_sheet(META, final=FINAL)    # share work_p2/human_eval/pair_*.png + rating_sheet_TEMPLATE.csv with 3 raters
# from p2_finetune.human_eval import win_rate; win_rate("work_p2/human_eval", ["rater1.csv", "rater2.csv", "rater3.csv"])

### Failure modes (>= 2, with images)

In [ ]:
cand = E.failure_candidates(clip, META)
for m, rows in cand.items():
    print(m)
    for score, f, pr in rows: print("  ", round(score, 1), f, "|", pr[:70])
fig, ax = plt.subplots(1, 3, figsize=(10, 4))
for a, (m, rows) in zip(ax, cand.items()):
    a.imshow(Image.open(rows[0][1])); a.set_title(f"{m} (lowest CLIP): {rows[0][2][:40]}", fontsize=6); a.axis("off")
plt.savefig("p2_failures.png", dpi=120); plt.show()

## C4 safety: license, NSFW filter, memorization

In [ ]:
print("images flagged by the SD safety checker:", res["nsfw_flagged"].to_dict())
top3, allpairs = E.memorization(clip, META, FINAL, train_paths)
fig, ax = plt.subplots(3, 2, figsize=(6, 9))
for r, p in enumerate(top3):
    ax[r, 0].imshow(Image.open(p["gen"])); ax[r, 1].imshow(Image.open(p["train"]))
    ax[r, 0].set_title(f"generated (cos={p['cosine']:.3f})", fontsize=7); ax[r, 1].set_title("closest training image", fontsize=7)
    ax[r, 0].axis("off"); ax[r, 1].axis("off")
plt.savefig("p2_memorization.png", dpi=120); plt.show()